# Setup

### Imports

In [1]:
# ---- Standard libraries ----
import os
import sys
import json
import csv
import time
import math
import ast
import copy
import heapq
import random
import pickle
import logging
from datetime import datetime
from collections import defaultdict, namedtuple, deque
from itertools import permutations

# ---- Data manipulation / visualization ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm import tqdm

# ---- Geo-related ----
import geojson
import geopandas as gpd
from shapely.geometry import Point, Polygon
from alphashape import alphashape
import geopy.distance
import networkx as nx
import requests
import overpass  # Overpass API for OpenStreetMap queries

# ---- PyTorch (for RL/deep learning parts) ----
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ---- Removed: this line (was for Colab only) ----
# from google.colab import drive


In [2]:
state_name = "Ohio"
city_name = "Columbus"
mini = True

### Paths

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

# Local data folder
DATA_DIR = PROJECT_ROOT / "Delivery_Data"
DATA_DIR.mkdir(exist_ok=True)

print(f"Using data directory: {DATA_DIR}")

Using data directory: /root/UMST_approach/Delivery_Data


In [4]:
# --- 5. Set Directories (local/cloud version) ---
from pathlib import Path
import os

try:
    # Step 1: base path (one level up from "Source Code/")
    personal_dir = str(Path.cwd().parent / "Delivery_Data") + "/"

    # Step 2: data folder based on city_name and mini flag
    data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

    # Step 3: output folder (same structure as before)
    output_dir = data_dir + "/UMST Graph/maddpg_baseline"

    # Step 4: create output directory if missing
    os.makedirs(output_dir, exist_ok=True)

    # Step 5: print info
    print(f"Data directory: {data_dir}")
    print(f"Output directory: {output_dir}")

    # Step 6: list contents for sanity check
    data = os.listdir(data_dir)
    print(f"Files in data_dir: {data}")

except Exception as e:
    print(f"Error setting up directories: {e}")
    print(f"Personal Dir Path: {personal_dir}")
    print(f"Data Dir Path: {data_dir}")

Data directory: /root/UMST_approach/Delivery_Data/Columbus_mini - RL Delivery Data
Output directory: /root/UMST_approach/Delivery_Data/Columbus_mini - RL Delivery Data/UMST Graph/maddpg_baseline
Files in data_dir: ['Images', 'Deliveries', 'Processed Location Data', 'Q Tables', 'avg_hotspot_data.json', 'Census Data', 'results_fixed_buffer', 'results_multiply', 'gh_cache', 'UMST Graph', 'Original Location Data', 'results_rangewise', 'results_randomized', 'Hotspot Data']


In [5]:
graph_path = data_dir + "/UMST Graph/graphs"
full_graph = graph_path + "/gh_hotspot_graph.graphml.xml"

### Sanity Check

In [6]:
import networkx as nx
import numpy as np

def verify_hotspot_graph(graph_path):
    """
    Load and verify the hotspot graph.
    Ensures node and edge attributes are valid and consistent.
    Returns: nx.Graph object if valid.
    """
    print(f"\n📂 Loading graph from: {graph_path}")
    try:
        G = nx.read_graphml(graph_path)
    except Exception as e:
        raise ValueError(f"❌ Failed to load graph: {e}")

    # Basic info
    print(f"✓ Graph loaded successfully.")
    print(f"   → {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # --- Node checks ---
    missing_latlon = [
        n for n, d in G.nodes(data=True)
        if 'lat' not in d or 'lon' not in d
    ]
    if missing_latlon:
        raise ValueError(f"❌ Some nodes missing lat/lon: {missing_latlon[:5]}")
    else:
        print(f"✓ All nodes have lat/lon attributes.")

    # --- Edge checks ---
    bad_edges = []
    distances, times = [], []
    for u, v, d in G.edges(data=True):
        dist = float(d.get('distance', np.nan))
        time = float(d.get('time', np.nan))
        if np.isnan(dist) or np.isnan(time) or dist <= 0 or time <= 0:
            bad_edges.append((u, v, d))
        else:
            distances.append(dist)
            times.append(time)

    if bad_edges:
        print(f"⚠️ Found {len(bad_edges)} edges with invalid data. Showing first 3:")
        for e in bad_edges[:3]:
            print("   ", e)
    else:
        print(f"✓ All edges have valid distance/time attributes.")

    print(f"📊 Mean distance: {np.mean(distances):.2f} km")
    print(f"📊 Mean travel time: {np.mean(times):.1f} mins")
    sample_edge = list(G.edges(data=True))[0]
    print(f"🔍 Sample edge: {sample_edge}")

    return G


In [7]:
verify_hotspot_graph(full_graph)


📂 Loading graph from: /root/UMST_approach/Delivery_Data/Columbus_mini - RL Delivery Data/UMST Graph/graphs/gh_hotspot_graph.graphml.xml
✓ Graph loaded successfully.
   → 26 nodes, 325 edges
✓ All nodes have lat/lon attributes.
✓ All edges have valid distance/time attributes.
📊 Mean distance: 4.88 km
📊 Mean travel time: 7.7 mins
🔍 Sample edge: ('39049001110', '39049003000', {'weight': 4.329004393965565, 'distance': 6.111, 'time': 10.083333333333334})


# Functions
Includes classes for:
- Generating deleviries
- 

In [ ]:
class Delivery:
    """
    Represents a single delivery task (pickup → dropoff).
    Tracks its status, assigned agent, and progress.
    """

    def __init__(self, delivery_id, start_node, end_node,
                 shortest_path, total_distance, total_time):
        self.id = delivery_id
        self.start_node = start_node
        self.end_node = end_node
        self.shortest_path = shortest_path
        self.total_distance = total_distance
        self.total_time = total_time

        # Dynamic state
        self.current_node = start_node
        self.picked = False
        self.delivered = False
        self.carrier_id = None

    # ---------------------------------------------------------
    # --- State transition helpers ---
    # ---------------------------------------------------------
    def mark_picked(self, agent_id):
        """Mark as picked by an agent."""
        self.picked = True
        self.carrier_id = agent_id
        self.current_node = self.start_node

    def mark_delivered(self):
        """Mark as delivered and clear carrier."""
        self.delivered = True
        self.carrier_id = None
        self.current_node = self.end_node

    def reset(self):
        """Reset to initial state for a new episode."""
        self.picked = False
        self.delivered = False
        self.carrier_id = None
        self.current_node = self.start_node

    def is_available(self):
        """Return True if not picked yet."""
        return not self.picked and not self.delivered

    def is_active(self):
        """Return True if picked but not yet delivered."""
        return self.picked and not self.delivered

    def __repr__(self):
        status = (
            "delivered" if self.delivered else
            "picked" if self.picked else
            "waiting"
        )
        return f"Delivery[{self.id}] {self.start_node}→{self.end_node} ({status})"

In [ ]:
save_path = output_dir + "/deliveries.pkl" # output_dir = data_dir + "/UMST Graph/maddpg_baseline"
import heapq, random, math, pickle, os
from tqdm import tqdm

class DeliveryList:
    """
    Generates and manages all delivery requests for training/testing.
    Fixed: iterates actual node keys (strings) from adjacency_matrix.
    """

    def __init__(self, adjacency_matrix, node_positions,
                 num_deliveries=1000, max_delivery_time=1800,
                 save_path=save_path, regenerate=False):
        """
        adjacency_matrix: dict[node_id] -> list of (neighbor_node_id, dist_km, time_minutes)
        node_positions: dict[node_id] -> (lat, lon)
        max_delivery_time: seconds (filter)
        """
        self.adjacency_matrix = adjacency_matrix
        self.node_positions = node_positions
        self.num_deliveries = num_deliveries
        self.max_delivery_time = max_delivery_time
        self.save_path = save_path

        # Load or generate deliveries
        if not regenerate and os.path.exists(save_path):
            print(f"📦 Loading existing deliveries from {save_path}")
            with open(save_path, "rb") as f:
                self.deliveries = pickle.load(f)
            print(f"✅ Loaded {len(self.deliveries)} deliveries.")
        else:
            print("🧮 Generating new deliveries...")
            # Use actual node keys from adjacency
            self.nodes = list(self.adjacency_matrix.keys())
            if not self.nodes:
                raise ValueError("Adjacency matrix appears empty!")

            self.precomputed_paths = self._precompute_paths()
            if not self.precomputed_paths:
                raise ValueError("No valid start→end pairs found within max_delivery_time! Check time units or adjacency.")

            self.deliveries = self._generate_deliveries()
            with open(save_path, "wb") as f:
                pickle.dump(self.deliveries, f)
            print(f"💾 Saved {len(self.deliveries)} deliveries to {save_path}")

    # ---------------------------------------------------------
    # A* shortest path (works with node IDs in adjacency)
    # ---------------------------------------------------------
    def _astar(self, start, goal):
        def heuristic(u, v):
            if u not in self.node_positions or v not in self.node_positions:
                return 0
            (x1, y1), (x2, y2) = self.node_positions[u], self.node_positions[v]
            return math.hypot(x1 - x2, y1 - y2)

        frontier = [(0, start)]
        came_from = {start: None}
        cost_so_far = {start: 0}

        while frontier:
            _, current = heapq.heappop(frontier)
            if current == goal:
                break

            for neighbor, dist_km, time_min in self.adjacency_matrix.get(current, []):
                time_sec = float(time_min) * 60.0  # convert minutes -> seconds
                new_cost = cost_so_far[current] + time_sec
                if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                    cost_so_far[neighbor] = new_cost
                    priority = new_cost + heuristic(neighbor, goal)
                    heapq.heappush(frontier, (priority, neighbor))
                    came_from[neighbor] = current

        if goal not in came_from:
            return float('inf'), []

        # Reconstruct path
        path, node = [], goal
        while node is not None:
            path.append(node)
            node = came_from[node]
        path.reverse()
        return cost_so_far[goal], path

    # ---------------------------------------------------------
    # Precompute all paths under the max_delivery_time
    # ---------------------------------------------------------
    def _precompute_paths(self):
        valid_pairs = {}
        nodes = self.nodes
        total_pairs = len(nodes) * (len(nodes) - 1)
        print(f"🗺️ Precomputing all node-to-node paths (filtered by time) for {len(nodes)} nodes ({total_pairs} pairs)...")

        with tqdm(total=total_pairs) as pbar:
            for start in nodes:
                for end in nodes:
                    if start == end:
                        pbar.update(1)
                        continue
                    travel_time, path = self._astar(start, end)
                    if travel_time <= self.max_delivery_time:
                        valid_pairs[(start, end)] = (travel_time, path)
                    pbar.update(1)

        print(f"✅ Found {len(valid_pairs)} valid node pairs within {self.max_delivery_time}s.")
        return valid_pairs

    # ---------------------------------------------------------
    # Generate deliveries
    # ---------------------------------------------------------
    def _path_total_distance(self, path):
        """Sum edge distances along a path (using adjacency)."""
        total = 0.0
        for i in range(len(path) - 1):
            a, b = path[i], path[i + 1]
            found = False
            for neigh, dist_km, _ in self.adjacency_matrix.get(a, []):
                if neigh == b:
                    total += float(dist_km)
                    found = True
                    break
            if not found:
                # missing edge data: treat as 0 or raise
                # We'll treat as 0 but print a warning (shouldn't happen)
                print(f"⚠️ Warning: edge {a}->{b} not found when computing distance for path.")
        return total

    def _generate_deliveries(self):
        from collections import Counter
        deliveries = []
        pair_keys = list(self.precomputed_paths.keys())
        if not pair_keys:
            raise ValueError("No valid (start,end) pairs available for delivery generation.")

        # track distribution for debug
        counter = Counter()

        for i in tqdm(range(self.num_deliveries), desc="Creating deliveries"):
            start, end = random.choice(pair_keys)
            travel_time, path = self.precomputed_paths[(start, end)]
            total_distance = self._path_total_distance(path)  # km
            d = Delivery(
                delivery_id=i,
                start_node=start,
                end_node=end,
                shortest_path=path,
                total_distance=total_distance,
                total_time=travel_time
            )
            deliveries.append(d)
            counter[(start, end)] += 1

        # debug: show top 5 frequent pairs
        most_common = counter.most_common(5)
        return deliveries

    # ---------------------------------------------------------
    # Helpers
    # ---------------------------------------------------------
    def sample(self, n):
        """Return a random subset of deliveries (fresh copies if needed)."""
        return random.sample(self.deliveries, min(n, len(self.deliveries)))

    def reset_all(self):
        """Reset all deliveries to their initial state."""
        for d in self.deliveries:
            d.reset()


Uncomment the first part to generate new deliveries

In [ ]:
# delivery_list = DeliveryList(
#     adjacency_matrix=adjacency_matrix,
#     node_positions=node_positions,
#     num_deliveries=1000,
#     max_delivery_time=1800,   # 30 min limit (in seconds)
#     save_path=save_path,
#     regenerate=True            # ✅ Set to True once, then False later
# )

reloaded = DeliveryList(
    adjacency_matrix=adjacency_matrix,
    node_positions=node_positions,
    save_path=save_path,
    regenerate=False
)

# New